# 银行低 PB 策略

每日筛选市值前十银行中 PB < 0.6 的标的，按回撤补仓并在现金不足时换股。

In [1]:
import os, sys, warnings
from pathlib import Path
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')
_ROOT = Path.cwd().resolve()
while not (_ROOT / 'pyproject.toml').exists():
    _ROOT = _ROOT.parent
os.chdir(_ROOT)
sys.path.insert(0, str(_ROOT))
from strategy.bank_pb import BANK_STOCKS, load_bank_data, screen_top10_banks, BankPBBacktest
from common.paths import DATA_DIR
print(f'银行板块股票数量: {len(BANK_STOCKS)}')

银行板块股票数量: 33


In [2]:
financial_cache, all_market_data, valid_banks = load_bank_data()
print(f'已预加载 {len(financial_cache)} 只银行股财务数据')
print(f'有效银行股数量: {len(valid_banks)}')
for c, n in sorted(valid_banks.items()):
    print(f'  {c} {n}')

已预加载 33 只银行股财务数据
有效银行股数量: 21
  000001 平安银行
  002142 宁波银行
  600016 民生银行
  600036 招商银行
  600919 江苏银行
  600926 杭州银行
  601009 南京银行
  601077 渝农商行
  601166 兴业银行
  601169 北京银行
  601229 上海银行
  601288 农业银行
  601328 交通银行
  601398 工商银行
  601658 邮储银行
  601818 光大银行
  601838 成都银行
  601939 建设银行
  601963 重庆银行
  601988 中国银行
  601998 中信银行


In [3]:
test_date = pd.Timestamp('2024-06-15')
q = screen_top10_banks(test_date, all_market_data, valid_banks, financial_cache)
print(f'{test_date.date()} 满足条件: {len(q)} 只')
for s in q:
    print(f"  {s['code']} {s['name']}: PB={s['pb']:.3f}, 市值={s['market_cap']/1e8:.0f}亿")

2024-06-15 满足条件: 0 只


In [4]:
backtest = BankPBBacktest(all_market_data, valid_banks, financial_cache, initial_capital=1_000_000)
backtest.run('2023-01-01', '2026-08-22')

回测期间: 2023-01-01 ~ 2026-08-22, 共 876 个交易日
  进度: 100/876, 净值: 1,053,660, 持仓: 9


  进度: 200/876, 净值: 1,052,057, 持仓: 9


  进度: 300/876, 净值: 1,176,460, 持仓: 8
  进度: 400/876, 净值: 1,402,539, 持仓: 6


  进度: 500/876, 净值: 1,447,601, 持仓: 4


  进度: 600/876, 净值: 1,766,554, 持仓: 4
  进度: 700/876, 净值: 1,722,085, 持仓: 2
  进度: 800/876, 净值: 1,586,257, 持仓: 2
回测完成! 最终净值: 1,609,278, 总交易: 123 笔


In [5]:
def load_benchmark(code, start, end):
    for suffix in ['_qfq.csv', '_market.csv']:
        p = DATA_DIR / f'{code}{suffix}'
        if p.exists():
            df = pd.read_csv(p)
            df['date'] = pd.to_datetime(df['date'])
            df.set_index('date', inplace=True)
            return df[(df.index >= start) & (df.index <= end)]
    return None

hs300 = load_benchmark('000300', '2023-01-01', '2026-08-22')
bank_etf = load_benchmark('512800', '2023-01-01', '2026-08-22')
print(f"沪深300: {len(hs300) if hs300 is not None else 0} 行, 银行ETF: {len(bank_etf) if bank_etf is not None else 0} 行")

沪深300: 881 行, 银行ETF: 0 行


In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

nav_df = pd.DataFrame(backtest.nav_history).set_index('date')
nav_df['return'] = nav_df['nav'] / nav_df['nav'].iloc[0] - 1
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
ax1 = axes[0, 0]
ax1.plot(nav_df.index, nav_df['nav'] / 1_000_000, label='银行低PB策略', linewidth=2)
if hs300 is not None:
    ax1.plot(hs300.index, hs300['close'] / hs300['close'].iloc[0], label='沪深300', alpha=0.7)
if bank_etf is not None:
    ax1.plot(bank_etf.index, bank_etf['close'] / bank_etf['close'].iloc[0], label='银行ETF', alpha=0.7)
ax1.set_title('净值曲线'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2 = axes[0, 1]
dd = (nav_df['nav'] - nav_df['nav'].cummax()) / nav_df['nav'].cummax()
ax2.fill_between(dd.index, dd, 0, alpha=0.5, color='red'); ax2.set_title('回撤'); ax2.grid(True, alpha=0.3)
ax3 = axes[1, 0]
ax3.bar(nav_df.index, nav_df['n_positions'], alpha=0.7); ax3.set_title('持仓数量'); ax3.grid(True, alpha=0.3)
ax4 = axes[1, 1]
ax4.plot(nav_df.index, nav_df['cash'] / nav_df['nav'], alpha=0.7); ax4.set_title('现金比例'); ax4.grid(True, alpha=0.3)
plt.tight_layout()
os.makedirs('reports/generated', exist_ok=True)
plt.savefig('reports/generated/bank_pb_strategy.png', dpi=150, bbox_inches='tight')
plt.show()
print('图表已保存')

图表已保存


In [7]:
from backtest.metrics import formatted_perf_row
results = [formatted_perf_row(nav_df['nav'], '银行低PB策略')]
if hs300 is not None: results.append(formatted_perf_row(hs300['close'], '沪深300'))
if bank_etf is not None: results.append(formatted_perf_row(bank_etf['close'], '银行ETF'))
perf_df = pd.DataFrame(results)
print(perf_df.to_string(index=False))

     策略   累计收益   年化收益   年化波动 Sharpe    最大回撤 Calmar
银行低PB策略 60.95% 14.09% 17.93%   0.79 -17.73%   0.79
  沪深300 18.80%  4.86% 17.55%   0.28 -24.80%   0.20


In [8]:
trades_df = pd.DataFrame(backtest.trade_history)
print(f'交易记录 (共 {len(trades_df)} 笔):')
if not trades_df.empty:
    print(trades_df[['date','code','name','action','price','shares','amount','reason']].to_string())

交易记录 (共 123 笔):
          date    code  name action  price  shares     amount  reason
0   2023-01-03  601398  工商银行    BUY   3.54   14100    49914.0    首次建仓
1   2023-01-03  601939  建设银行    BUY   4.60   10800    49680.0    首次建仓
2   2023-01-03  601288  农业银行    BUY   2.39   20900    49951.0    首次建仓
3   2023-01-03  601988  中国银行    BUY   2.61   19100    49851.0    首次建仓
4   2023-01-03  601658  邮储银行    BUY   3.83   13000    49790.0    首次建仓
5   2023-01-03  601328  交通银行    BUY   3.83   13000    49790.0    首次建仓
6   2023-01-03  601166  兴业银行    BUY  14.02    3500    49070.0    首次建仓
7   2023-01-03  601998  中信银行    BUY   4.08   12200    49776.0    首次建仓
8   2023-03-09  000001  平安银行    BUY  10.82    4100    44362.0    首次建仓
9   2023-05-12  601658  邮储银行    BUY   4.31    5100    21981.0  补仓跌11%
10  2023-05-15  601998  中信银行    BUY   5.77    3800    21926.0  补仓跌11%
11  2023-05-16  601998  中信银行    BUY   5.70    3800    21660.0  补仓跌12%
12  2023-05-17  601658  邮储银行    BUY   4.34    5100    22134.0  补仓跌11%
13  

In [9]:
os.makedirs('reports/generated', exist_ok=True)
nav_df.to_csv('reports/generated/bank_pb_nav.csv')
if not trades_df.empty:
    trades_df.to_csv('reports/generated/bank_pb_trades.csv', index=False)
perf_df.to_csv('reports/generated/bank_pb_performance.csv', index=False)
print('结果已保存')

结果已保存
